In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"C:\Users\Retaj\Downloads\archive (10)\student_exam_performance.csv")

print("Dataset shape:", df.shape)  # Basic dataset check

duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

print("Number of duplicated student IDs:", df["student_id"].duplicated().sum())
print("Number of unique student IDs:", df["student_id"].nunique())


Dataset shape: (100000, 44)
Number of duplicate rows: 0
Number of duplicated student IDs: 0
Number of unique student IDs: 100000


In [2]:
# Check the target before the split
print("\nMissing target values:")
print(df["pass_status"].isna().sum())

print("\nTarget classes:")
print(df["pass_status"].unique())

print("\nTarget distribution:")
print(df["pass_status"].value_counts())

print("\nTarget distribution (%):")
print(df["pass_status"].value_counts(normalize=True) * 100)


Missing target values:
0

Target classes:
<ArrowStringArray>
['Pass', 'Fail']
Length: 2, dtype: str

Target distribution:
pass_status
Pass    77389
Fail    22611
Name: count, dtype: int64

Target distribution (%):
pass_status
Pass    77.389
Fail    22.611
Name: proportion, dtype: float64


In [3]:
target_mapping = {"Pass": 0, "Fail": 1}  # Make Fail the positive class
df["pass_status"] = df["pass_status"].map(target_mapping)

print("Encoded target distribution:")
print(df["pass_status"].value_counts())


# Remove the target, ID, and outcome-related columns
target_column = "pass_status"
leakage_columns = ["student_id","questions_attempted", "questions_correct", "exam_score", "performance_grade", "performance_level"]

X = df.drop(columns=[target_column] + leakage_columns)
y = df[target_column]

Encoded target distribution:
pass_status
0    77389
1    22611
Name: count, dtype: int64


In [4]:

# Clean categorical values
categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns

for column in categorical_columns:
    X[column] = X[column].astype("string").str.strip().str.lower()


# Identify numerical and categorical columns
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns

print("Numerical columns:")
print(list(numerical_columns))

print("\nCategorical columns:")
print(list(categorical_columns))

Numerical columns:
['age', 'previous_exam_score', 'previous_gpa', 'attendance_percentage', 'assignment_completion_rate', 'study_hours_per_day', 'self_study_hours', 'private_tuition', 'online_learning_hours', 'practice_tests_completed', 'sleep_hours', 'daily_screen_time', 'physical_activity_hours', 'stress_level', 'internet_access', 'online_course_hours', 'exam_preparation_days', 'time_management_score', 'exam_anxiety_level']

Categorical columns:
['gender', 'education_level', 'school_type', 'family_income', 'parent_education', 'urban_rural', 'class_participation', 'study_consistency', 'study_environment', 'study_method', 'revision_frequency', 'notes_quality', 'sleep_quality', 'break_frequency', 'motivation_level', 'device_availability', 'educational_app_usage', 'exam_difficulty']


In [5]:
# Split before imputation to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))


# Check important numerical ranges for possible invalid values
columns_to_check = ["age", "previous_exam_score", "previous_gpa", "attendance_percentage", "assignment_completion_rate", "study_hours_per_day", "self_study_hours", "online_course_hours", "physical_activity_hours", "daily_screen_time"]

print("\nRange checks:")

for column in columns_to_check:
    if column in X_train.columns:
        print(column, "min =", X_train[column].min(), "max =", X_train[column].max())

Training target distribution:
pass_status
0    0.773887
1    0.226112
Name: proportion, dtype: float64

Testing target distribution:
pass_status
0    0.7739
1    0.2261
Name: proportion, dtype: float64

Range checks:
age min = 14 max = 20
previous_exam_score min = 30.0 max = 100.0
previous_gpa min = 1.0 max = 4.0
attendance_percentage min = 46.34 max = 100.0
assignment_completion_rate min = 20.0 max = 100.0
study_hours_per_day min = 0.5 max = 12.0
self_study_hours min = 0.25 max = 10.0
online_course_hours min = 0.0 max = 20.0
physical_activity_hours min = 0.0 max = 12.0
daily_screen_time min = 1.0 max = 11.94


In [6]:
# Fill numerical missing values using the training median
for column in numerical_columns:
    median_value = X_train[column].median()
    X_train[column] = X_train[column].fillna(median_value)
    X_test[column] = X_test[column].fillna(median_value)


# Fill categorical missing values using the training mode
for column in categorical_columns:
    mode_value = X_train[column].mode()[0]
    X_train[column] = X_train[column].fillna(mode_value)
    X_test[column] = X_test[column].fillna(mode_value)


print("Missing values after imputation:")
print("Training:", X_train.isnull().sum().sum())
print("Testing:", X_test.isnull().sum().sum())

Missing values after imputation:
Training: 0
Testing: 0


In [7]:
# Convert categorical variables into 0/1 columns
X_train = pd.get_dummies(X_train, columns=categorical_columns, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_columns, drop_first=True, dtype=int)

# Make sure train and test have exactly the same features
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


print("Shape after one-hot encoding:")
print("Training:", X_train.shape)
print("Testing:", X_test.shape)


# Keep unscaled data for tree-based models
# Create scaled copies for models that are sensitive to feature scale
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

Shape after one-hot encoding:
Training: (80000, 60)
Testing: (20000, 60)


In [8]:

# Standardize only the original numerical variables
scaler = StandardScaler()

scaler_columns = [column for column in numerical_columns if column in X_train.columns]

X_train_scaled[scaler_columns] = scaler.fit_transform(X_train[scaler_columns])
X_test_scaled[scaler_columns] = scaler.transform(X_test[scaler_columns])


In [9]:

# Final checks
print("Pre-processing completed successfully.")

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nScaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)

print("\nTarget variable distribution:")
print(y.value_counts())

print("\nMissing values in training data:")
print(X_train.isnull().sum().sum())

print("Missing values in testing data:")
print(X_test.isnull().sum().sum())

print("\nMissing values in scaled training data:")
print(X_train_scaled.isnull().sum().sum())

print("Missing values in scaled testing data:")
print(X_test_scaled.isnull().sum().sum())

print("\nFinal number of features:", X_train.shape[1])

Pre-processing completed successfully.

Training data shape: (80000, 60)
Testing data shape: (20000, 60)

Scaled training data shape: (80000, 60)
Scaled testing data shape: (20000, 60)

Target variable distribution:
pass_status
0    77389
1    22611
Name: count, dtype: int64

Missing values in training data:


0
Missing values in testing data:
0

Missing values in scaled training data:
0
Missing values in scaled testing data:
0

Final number of features: 60
